In [3]:
import torch 
import torch.nn as nn
import torch.optim as optim

from torchvision.transforms import transforms
from torchvision import datasets
from torch.utils.data import DataLoader,random_split

import matplotlib.pyplot as plt

In [4]:
device = torch.device('cpu')

In [5]:
transform = transforms.Compose([
    transforms.Resize((28,28)),
    transforms.ToTensor()
])

In [7]:
dataset = datasets.ImageFolder(
    r"C:\Dl\PYTORCH_PROJECT_04\BreaKHis_v1\histology_slides\breast",
    transform=transform
)


print(dataset.classes, len(dataset))

['benign', 'malignant'] 4090


In [11]:
import os, random, shutil
from collections import defaultdict

SRC = r"C:\Dl\PYTORCH_PROJECT_04\BreaKHis_v1\histology_slides\breast"
DST = r"C:\Dl\PYTORCH_PROJECT_04\split_data"

# Step 1: group all images by patient ID
patients = defaultdict(list)

for dirpath, _, files in os.walk(SRC):
    for file in files:
        if file.endswith(".png"):
            # filename looks like: SOB_B_A-14-22549AB-40-001.png
            parts = file.split("-")
            patient_id = parts[1]          # e.g. "14"
            label = "benign" if "_B_" in file else "malignant"
            patients[patient_id].append((os.path.join(dirpath, file), label))

# Step 2: shuffle patients (not images!) and split 70/30
patient_ids = list(patients.keys())
random.seed(42)
random.shuffle(patient_ids)

split_point = int(0.7 * len(patient_ids))
train_patients = patient_ids[:split_point]
test_patients = patient_ids[split_point:]

# Step 3: copy each patient's images into train or test folder
for pid in train_patients:
    for filepath, label in patients[pid]:
        out = os.path.join(DST, "train_data", label)
        os.makedirs(out, exist_ok=True)
        shutil.copy2(filepath, out)

for pid in test_patients:
    for filepath, label in patients[pid]:
        out = os.path.join(DST, "test_data", label)
        os.makedirs(out, exist_ok=True)
        shutil.copy2(filepath, out)

print("Train patients:", len(train_patients), "Test patients:", len(test_patients))

# Step 4: NOW load the folders it just created
from torchvision import datasets

train_dataset = datasets.ImageFolder(os.path.join(DST, "train_data"))
test_dataset = datasets.ImageFolder(os.path.join(DST, "test_data"))

print(train_dataset.classes, len(train_dataset), len(test_dataset))

OSError: [Errno 28] No space left on device

In [14]:
import shutil
total, used, free = shutil.disk_usage(r"C:\Dl")
print(f"Free: {free / (1024**3):.2f} GB")

Free: 0.00 GB
